# Calendar Studio · desde cero

La librería es independiente de Model Arena. Este ejemplo usa un fixture explícito de festivos del proyecto, no un servicio de festivos oficiales.

In [ ]:
import sys,subprocess
from importlib.metadata import version,PackageNotFoundError
try: installed=version("calendar-studio")
except PackageNotFoundError: installed=None
if installed!="0.1.0":
    subprocess.run([sys.executable,"-m","pip","install","--no-cache-dir","calendar-studio==0.1.0"],check=True)
from calendar_studio import CalendarStudio,CalendarSpec,CalendarArtifact,example_holidays


## 1. Configurar reglas generales

In [ ]:
spec=CalendarSpec(name="operaciones",timezone="America/Mexico_City",rules=[
    dict(type="monthly_event",name="cobro",days=[15,30],adjustment="preceding"),
    dict(type="monthly_event",name="pago",days=[17],adjustment="following"),
    dict(type="business_offsets",events=["cobro","pago"],before=10,after=10),
])
calendar=CalendarStudio(spec).generate(start="2026-01-01",end="2026-12-31",holidays=example_holidays())
calendar.validate().raise_for_errors()
calendar.to_pandas().head()


## 2. Guardar y volver a leer

La carpeta no debe existir; se conserva la configuración y la firma de los datos.

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
output=Path("calendars")/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
calendar.save(output)
restored=CalendarArtifact.load(output)
restored.view("business").to_pandas().head()


## 3. Perfil histórico MAC

Se conserva como perfil separado. Incluye la corrección del 2 de noviembre de 2026 cuando se utiliza el fixture actualizado. Esto no modifica Model Arena ni sus corridas existentes.

In [ ]:
legacy=CalendarStudio.from_profile("mac_legacy_v1").generate(
    start="2026-01-01",end="2026-12-31",holidays=example_holidays())
mac=legacy.view("mac_legacy_v1")
print("Variables:",len(mac.to_pandas().columns)-1)
mac.export(output/"mac_business.parquet")
mac.to_pandas().head()
